In [1]:
import os
import re
import time
import mimetypes
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from urllib.parse import urlparse
from datetime import datetime, timezone
import csv
import pandas as pd
import requests
from typing import Any

In [4]:
# =========================
# CONFIG  ← update these paths to match your setup
# =========================
INPUT_POSTS_CSV = "Threads/threads_posts_parsed.csv"   # your Threads CSV
OUT_DIR         = Path("Threads/post_info/Downloads")

# persistent job log (appended over time, crash-safe)
JOB_LOG_CSV = "Threads/posts_downloads_log.csv"

SLEEP_SECONDS = 0.2
TIMEOUT       = 60
MAX_RETRIES   = 2

In [5]:
# =========================
# Helpers
# =========================
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def make_run_id() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


def safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    return re.sub(r"[^a-zA-Z0-9._-]", "", s)


def split_urls(s: Any) -> List[str]:
    """Split a comma-separated URL string; returns [] for blank/null cells."""
    if s is None:
        return []
    s = str(s).strip()
    if not s or s.lower() in {"nan", "none", "null", ""}:
        return []
    return [p.strip() for p in s.split(",") if p.strip()]


def is_valid_http_url(u: Any) -> bool:
    if u is None:
        return False
    u = str(u).strip()
    if not u or u.lower() in {"nan", "none", "null"}:
        return False
    return u.startswith("http://") or u.startswith("https://")


def guess_ext_from_url(url: str) -> str:
    path = urlparse(url).path
    ext = Path(path).suffix.lower()
    return ext if (ext and len(ext) <= 5) else ".bin"


def guess_ext_from_content_type(content_type: Optional[str], fallback: str = ".bin") -> str:
    if not content_type:
        return fallback
    content_type = content_type.split(";")[0].strip().lower()
    ext = mimetypes.guess_extension(content_type) or fallback
    if ext == ".jpe":
        ext = ".jpg"
    return ext



In [6]:
# =========================
# Build tasks from CSV
# =========================
# Priority logic (no post_type check needed):
#   1. If carousel_media_urls is non-blank  → download each URL as a carousel slide
#   2. Otherwise                            → download cover_url as a cover image

def build_tasks_from_csv(csv_path: str) -> List[Dict]:
    df = pd.read_csv(csv_path, dtype=str)  # read everything as strings
    df = df.fillna("")  # turn NaN → empty string

    # Make sure required columns exist (graceful fallback)
    for col in ["post_id", "carousel_media_urls", "cover_url"]:
        if col not in df.columns:
            df[col] = ""

    tasks: List[Dict] = []

    for _, r in df.iterrows():
        post_id = str(r.get("post_id", "")).strip()
        if not post_id:
            continue

        carousel_urls = split_urls(r.get("carousel_media_urls", ""))
        cover_url = str(r.get("cover_url", "") or "").strip()

        # --- Priority 1: carousel ---
        if carousel_urls:
            for i, u in enumerate(carousel_urls, start=1):
                if not is_valid_http_url(u):
                    continue
                tasks.append({
                    "post_id": post_id,
                    "slide_number": i,
                    "url": u,
                    "kind": "carousel_slide",
                    "file_title_base": f"{safe_filename(post_id)}_{i}",
                })
        # --- Priority 2: cover_url fallback ---
        elif is_valid_http_url(cover_url):
            tasks.append({
                "post_id": post_id,
                "slide_number": 1,
                "url": cover_url,
                "kind": "cover_image",
                "file_title_base": safe_filename(post_id),
            })
        # else: nothing to download for this post

    return tasks



In [7]:
# =========================
# Persistent job log
# =========================
JOB_LOG_COLUMNS = [
    "run_id", "run_started_at", "run_finished_at",
    "post_id", "slide_number", "kind", "url",
    "downloaded", "status_code",
    "error_type", "error_message", "error_bucket",
    "saved_filename", "local_path",
    "attempts", "duration_sec",
]


def load_job_log(job_log_csv: str) -> pd.DataFrame:
    if not Path(job_log_csv).exists():
        return pd.DataFrame(columns=JOB_LOG_COLUMNS)
    return pd.read_csv(job_log_csv, dtype=str).fillna("")


def task_key(task: Dict) -> Tuple[str, int, str, str]:
    return (str(task["post_id"]), int(task["slide_number"]), str(task["kind"]), str(task["url"]))


def filter_tasks_skip_success(tasks: List[Dict], job_log_df: pd.DataFrame) -> List[Dict]:
    """Skip tasks already successfully downloaded in a previous run."""
    if job_log_df.empty:
        return tasks

    jl = job_log_df.copy()
    if "downloaded" in jl.columns:
        jl["downloaded"] = jl["downloaded"].astype(str).str.lower().isin(["true", "1", "yes"])

    success = jl[jl.get("downloaded", False) == True]
    if success.empty:
        return tasks

    success_keys = set(zip(
        success["post_id"].astype(str),
        success["slide_number"].astype(int),
        success["kind"].astype(str),
        success["url"].astype(str),
    ))
    return [t for t in tasks if task_key(t) not in success_keys]


def append_job_log(job_log_csv: str, rows: List[Dict]) -> None:
    """Append rows to the CSV with a fixed schema (crash-safe, one row at a time)."""
    if not rows:
        return
    out_path = Path(job_log_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = out_path.exists() and out_path.stat().st_size > 0

    with out_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=JOB_LOG_COLUMNS, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in JOB_LOG_COLUMNS})
        f.flush()
        try:
            os.fsync(f.fileno())
        except Exception:
            pass



In [8]:
# =========================
# Download single file
# =========================
def download_one(session: requests.Session, url: str, out_path_no_ext: Path) -> Dict:
    """
    Returns dict with:
      ok, status_code, saved_path, saved_filename,
      error_type, error_message, attempts, duration_sec
    """
    t0 = time.time()

    url_ext = guess_ext_from_url(url)
    final_path = out_path_no_ext.with_suffix(url_ext)

    # Already exists locally → treat as success
    if final_path.exists() and final_path.stat().st_size > 0:
        return {
            "ok": True, "status_code": None,
            "saved_path": str(final_path), "saved_filename": final_path.name,
            "error_type": "", "error_message": "",
            "attempts": 0, "duration_sec": round(time.time() - t0, 4),
        }

    last_exc = None
    last_status = None

    for attempt in range(1, MAX_RETRIES + 2):  # total = MAX_RETRIES + 1
        try:
            with session.get(url, stream=True, timeout=TIMEOUT, allow_redirects=True) as resp:
                last_status = resp.status_code

                if resp.status_code == 429:
                    raise requests.HTTPError("HTTP 429 (rate limited)")
                if resp.status_code in (403, 404):
                    raise requests.HTTPError(f"HTTP {resp.status_code} (expired/private?)")

                resp.raise_for_status()

                # Infer extension from Content-Type if URL gave none
                if url_ext == ".bin":
                    ext2 = guess_ext_from_content_type(resp.headers.get("Content-Type"), fallback=".bin")
                    final_path = out_path_no_ext.with_suffix(ext2)

                # File appeared between our check and now
                if final_path.exists() and final_path.stat().st_size > 0:
                    return {
                        "ok": True, "status_code": last_status,
                        "saved_path": str(final_path), "saved_filename": final_path.name,
                        "error_type": "", "error_message": "",
                        "attempts": attempt, "duration_sec": round(time.time() - t0, 4),
                    }

                OUT_DIR.mkdir(parents=True, exist_ok=True)
                tmp_path = final_path.with_suffix(final_path.suffix + ".part")

                with tmp_path.open("wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f.write(chunk)

                tmp_path.replace(final_path)

                return {
                    "ok": True, "status_code": last_status,
                    "saved_path": str(final_path), "saved_filename": final_path.name,
                    "error_type": "", "error_message": "",
                    "attempts": attempt, "duration_sec": round(time.time() - t0, 4),
                }

        except Exception as e:
            last_exc = e
            time.sleep(0.5 * attempt)  # back-off

    err_type = type(last_exc).__name__ if last_exc else "UnknownError"
    err_msg = str(last_exc) if last_exc else "Unknown error"

    return {
        "ok": False, "status_code": last_status,
        "saved_path": "", "saved_filename": "",
        "error_type": err_type, "error_message": err_msg,
        "attempts": MAX_RETRIES + 1, "duration_sec": round(time.time() - t0, 4),
    }



In [10]:
# =========================
# Main runner
# =========================
def run_downloads_with_joblog(
        input_csv: str = INPUT_POSTS_CSV,
        out_dir: Path = OUT_DIR,
        job_log_csv: str = JOB_LOG_CSV,
        sleep_seconds: float = SLEEP_SECONDS,
) -> pd.DataFrame:
    """
    Reads the Threads CSV, builds download tasks, skips prior successes,
    downloads each file, and appends one log row per task (crash-safe).
    Returns a DataFrame of rows from THIS run only.
    """
    global OUT_DIR
    OUT_DIR = out_dir

    run_id = make_run_id()
    run_started_at = int(time.time())

    # Build tasks
    tasks = build_tasks_from_csv(input_csv)
    print(f"Total tasks built: {len(tasks)}")

    # Skip previously successful downloads
    job_log_df = load_job_log(job_log_csv)
    tasks_to_run = filter_tasks_skip_success(tasks, job_log_df)

    print(f"Skipping previously successful: {len(tasks) - len(tasks_to_run)}")
    print(f"Tasks to run now: {len(tasks_to_run)}")

    run_rows: List[Dict] = []

    with requests.Session() as session:
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/121.0.0.0 Safari/537.36",
            "Referer": "https://www.threads.net/",
            "Origin": "https://www.threads.net",
            "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Connection": "keep-alive",
        })

        for idx, t in enumerate(tasks_to_run, start=1):
            post_id = t["post_id"]
            slide_number = int(t["slide_number"])
            url = t["url"]
            kind = t["kind"]
            file_base = t["file_title_base"]

            out_path_no_ext = OUT_DIR / file_base
            result = download_one(session, url, out_path_no_ext)

            # Categorise errors for easy analysis
            error_bucket = ""
            if not result["ok"]:
                sc = result["status_code"]
                if sc in (403, 404):
                    error_bucket = "expired_or_private"
                elif sc == 429:
                    error_bucket = "rate_limited"
                elif result["error_type"] in ("MissingSchema", "InvalidURL"):
                    error_bucket = "bad_url"
                else:
                    error_bucket = "other"

            row = {
                "run_id": run_id,
                "run_started_at": run_started_at,
                "run_finished_at": "",  # filled at end
                "post_id": post_id,
                "slide_number": slide_number,
                "kind": kind,
                "url": url,
                "downloaded": result["ok"],
                "status_code": result["status_code"],
                "error_bucket": error_bucket,
                "error_type": result["error_type"],
                "error_message": result["error_message"],
                "saved_filename": result["saved_filename"],
                "local_path": result["saved_path"],
                "attempts": result["attempts"],
                "duration_sec": result["duration_sec"],
            }

            # ✅ Write to log immediately (crash-safe)
            append_job_log(job_log_csv, [row])
            run_rows.append(row)

            print(
                f"[{idx}/{len(tasks_to_run)}] post_id={post_id} slide={slide_number} "
                f"kind={kind} ok={result['ok']} status={result['status_code']} "
                f"{('err=' + row['error_type']) if not result['ok'] else ''}"
            )

            time.sleep(sleep_seconds)

    run_finished_at = int(time.time())
    for r in run_rows:
        r["run_finished_at"] = run_finished_at

    return pd.DataFrame(run_rows)

In [11]:


if __name__ == "__main__":
    df_result = run_downloads_with_joblog(
        input_csv=INPUT_POSTS_CSV,
        out_dir=OUT_DIR,
        job_log_csv=JOB_LOG_CSV,
        sleep_seconds=SLEEP_SECONDS,
    )
    print("\n--- Run summary ---")
    print(df_result[["post_id", "slide_number", "kind", "downloaded", "error_bucket"]]
          .value_counts("downloaded"))

Total tasks built: 0
Skipping previously successful: 0
Tasks to run now: 0

--- Run summary ---


KeyError: "None of [Index(['post_id', 'slide_number', 'kind', 'downloaded', 'error_bucket'], dtype='object')] are in the [columns]"